# Personalized Learning Path Recommender
## Strategy: TF-IDF + Cosine Similarity

In [ ]:
import pandas as pd
import numpy as np
import re
import time
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize

In [ ]:
# Load data
train = pd.read_csv('train.csv')
test  = pd.read_csv('test.csv')
print(f"Train: {train.shape}, Test: {test.shape}")

In [ ]:
# Text preprocessing
def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

train["clean_review"] = train["Reviews"].apply(clean_text)
test["clean_review"]  = test["Reviews"].apply(clean_text)

# Add course name to train features
train["feature_text"] = train["Course"].apply(clean_text) + " " + train["clean_review"]
test["feature_text"]  = test["clean_review"]

In [ ]:
# TF-IDF vectorization
vectorizer = TfidfVectorizer(
    max_features=30_000,
    ngram_range=(1, 2),
    sublinear_tf=True,
    min_df=2,
    max_df=0.95,
    strip_accents="unicode",
    analyzer="word",
)

train_tfidf = vectorizer.fit_transform(train["feature_text"])
test_tfidf  = vectorizer.transform(test["feature_text"])

# L2 normalization
train_tfidf = normalize(train_tfidf, norm="l2")
test_tfidf  = normalize(test_tfidf,  norm="l2")

print(f"TF-IDF matrix – train: {train_tfidf.shape}, test: {test_tfidf.shape}")

In [ ]:
# Generate top-10 recommendations
BATCH_SIZE = 500
TOP_K = 10
train_indices = train["Index"].values

results = []
n_test = test_tfidf.shape[0]

for start in range(0, n_test, BATCH_SIZE):
    end = min(start + BATCH_SIZE, n_test)
    batch = test_tfidf[start:end]
    
    # Cosine similarity
    sim_matrix = (batch @ train_tfidf.T).toarray()
    
    for i, row_sim in enumerate(sim_matrix):
        top_k_pos = np.argpartition(row_sim, -TOP_K)[-TOP_K:]
        top_k_pos = top_k_pos[np.argsort(row_sim[top_k_pos])[::-1]]
        top_k_indices = train_indices[top_k_pos].tolist()
        test_idx = test["Index"].iloc[start + i]
        results.append((test_idx, top_k_indices))

print(f"Generated {len(results)} recommendations")

In [ ]:
# Create submission
submission = pd.DataFrame(results, columns=["Index", "Index_list"])
submission["Index_list"] = submission["Index_list"].apply(lambda x: str(x))

print(f"Submission shape: {submission.shape}")
print(submission.head())

# Save
submission.to_csv('submission.csv', index=False)
print("Submission saved!")